In [1]:
!pip install -q pandas matplotlib scikit-learn transformers datasets accelerate


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from datasets import Dataset
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from transformers import TrainingArguments
from transformers import Trainer

from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import classification_report
from sklearn.metrics import ConfusionMatrixDisplay

d:\sentiment and sarcasm analysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("GPU is not available. Training will be slower on CPU.")

CUDA available: True
GPU name: NVIDIA GeForce RTX 4050 Laptop GPU


In [4]:
os.makedirs("../outputs/tables", exist_ok=True)
os.makedirs("../outputs/figures", exist_ok=True)
os.makedirs("../outputs/predictions", exist_ok=True)
os.makedirs("../outputs/models", exist_ok=True)

In [5]:
sentiment_df = pd.read_csv("../outputs/data/sentiment.csv")

print("Data shape:", sentiment_df.shape)
sentiment_df.head()

Data shape: (12601, 9)


,text,label,variety,source,task,split,text_length,word_count,clean_text
0,This was one of the best dishes I've EVER had!...,1,en-AU,Google,Sentiment,train,641,117,this was one of the best dishes i've ever had!...
1,This Mexican restaurant in Penrith is a great ...,1,en-AU,Google,Sentiment,train,576,93,this mexican restaurant in penrith is a great ...
2,"This was not to bad, I ordered the big pork ri...",1,en-AU,Google,Sentiment,train,309,62,"this was not to bad, i ordered the big pork ri..."
3,Clean cool and a nice smaller casino to check ...,1,en-AU,Google,Sentiment,train,251,46,clean cool and a nice smaller casino to check ...
4,Well set out. Great areas to enjoy. Good food ...,1,en-AU,Google,Sentiment,train,138,26,well set out. great areas to enjoy. good food ...


In [6]:
print("Label distribution:")
print(sentiment_df["label"].value_counts().sort_index())

print("\nSplit distribution:")
print(sentiment_df["split"].value_counts())

Label distribution:
label
0    6355
1    6246
Name: count, dtype: int64

Split distribution:
split
train         8866
test          2523
validation    1212
Name: count, dtype: int64


In [7]:
train_df = sentiment_df[sentiment_df["split"] == "train"].copy()
valid_df = sentiment_df[sentiment_df["split"] == "validation"].copy()
test_df = sentiment_df[sentiment_df["split"] == "test"].copy()

print("Train:", train_df.shape)
print("Validation:", valid_df.shape)
print("Test:", test_df.shape)

Train: (8866, 9)
Validation: (1212, 9)
Test: (2523, 9)


In [8]:
train_df = train_df[["text", "label"]]
valid_df = valid_df[["text", "label"]]
test_df = test_df[["text", "label", "variety", "source", "split"]]

print(train_df.head())

                                                text  label
0  This was one of the best dishes I've EVER had!...      1
1  This Mexican restaurant in Penrith is a great ...      1
2  This was not to bad, I ordered the big pork ri...      1
3  Clean cool and a nice smaller casino to check ...      1
4  Well set out. Great areas to enjoy. Good food ...      1


In [9]:
train_data = Dataset.from_pandas(train_df)
valid_data = Dataset.from_pandas(valid_df)
test_data = Dataset.from_pandas(test_df)

print(train_data)
print(valid_data)
print(test_data)

Dataset({
    features: ['text', 'label'],
    num_rows: 8866
})
Dataset({
    features: ['text', 'label'],
    num_rows: 1212
})
Dataset({
    features: ['text', 'label', 'variety', 'source', 'split'],
    num_rows: 2523
})


In [10]:
model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [11]:
max_length = 128

def tokenize_data(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=max_length
    )

In [12]:
train_data = train_data.map(tokenize_data, batched=True)
valid_data = valid_data.map(tokenize_data, batched=True)
test_data = test_data.map(tokenize_data, batched=True)

print(train_data[0].keys())

Map: 100%|██████████| 2523/2523 [00:00<00:00, 8438.64 examples/s]

dict_keys(['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'])


In [13]:
num_labels = sentiment_df["label"].nunique()

print("Number of labels:", num_labels)
print("Labels:", sorted(sentiment_df["label"].unique()))

Number of labels: 2
Labels: [np.int64(0), np.int64(1)]


In [14]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5925.34it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider tr

In [15]:
def compute_metrics(eval_result):
    logits, labels = eval_result
    predictions = np.argmax(logits, axis=1)
    
    accuracy = accuracy_score(labels, predictions)
    macro_f1 = f1_score(labels, predictions, average="macro")
    macro_precision = precision_score(labels, predictions, average="macro", zero_division=0)
    macro_recall = recall_score(labels, predictions, average="macro", zero_division=0)
    
    return {
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall
    }

In [16]:
use_fp16 = torch.cuda.is_available()

training_args = TrainingArguments(
    output_dir="../outputs/models/bert_sentiment_checkpoints",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="no",
    report_to="none",
    fp16=use_fp16
)

print("Using fp16:", use_fp16)

Using fp16: True


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=valid_data,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

TypeError: Trainer.__init__() got an unexpected keyword argument 'tokenizer'